# Paso 1: Preparación del archivo de datos

El primer paso consiste en generar un archivo en formato CSV que contenga los datos puntuales de balance de masa específico. Puedes usar excel, planillas de google o bloc de notas.

El archivo debe incluir las siguientes columnas:

- **POINT_ID**: Identificador único del punto de medición.
- **FROM_DATE**: Fecha de inicio del registro (formato: `YYYY-MM-DD`).
- **TO_DATE**: Fecha de fin del registro (formato: `YYYY-MM-DD`).
- **POINT_LON**: Latitud en grados decimales (WGS84).
- **POINT_LAT**: Longitud en grados decimales (WGS84).
- **POINT_ELEVATION**: Altitud en metros sobre el nivel del mar.
- **POINT_BALANCE**: Balance de masa específico (en mWE o unidad equivalente).

**Ejemplo para la Cordillera Blanca:**

Ve la siguiente captura de pantalla donde muestra un ejemplo de archivo CSV con datos simulados para glaciares de esta región.

In [ ]:
import pandas as pd
import os
import warnings
from tqdm.notebook import tqdm
import zipfile
import cdsapi
import zipfile
import numpy as np
import glob
import xarray as xr

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

In [ ]:
path_file = 'data/data-zona.csv'
pd.read_csv(path_file, sep=',') 

# Paso 2: Descargar datos del ERA5

In [ ]:
#path_ERA5_raw = './era5land/raw/'
path_ERA5_raw = './era5land-antarc/raw/'

In [ ]:
RUN = True
if RUN:
    os.makedirs(path_ERA5_raw, exist_ok=True)
    c = cdsapi.Client()
    c.retrieve(
        'reanalysis-era5-land-monthly-means', {
            'product_type': ['monthly_averaged_reanalysis'],
            "variable": [
                "2m_temperature",
                "forecast_albedo",
                "surface_latent_heat_flux",
                "surface_net_thermal_radiation",
                "surface_sensible_heat_flux",
                "surface_solar_radiation_downwards",
                "total_precipitation"
            ],
            "year": [ 
                "2000", "2001", "2002", "2003",
                "2004", "2005", "2006", "2007",
                "2008", "2009", "2010", "2011",
                "2012", "2013", "2014",
                "2015", "2016", "2017",
                "2018", "2019", "2020",
                "2021", "2022", "2023", 
                "2024",
            ],
            "month": [
                "01", "02", "03",
                "04", "05", "06",
                "07", "08", "09",
                "10", "11", "12"
            ],
            'time': ["00:00"],
            "data_format": "netcdf",
            "download_format": "zip",
            #"area": [-45, -74, -55, -69],
            "area": [-62, -58.5, -59, -62.5]
        }, path_ERA5_raw+'download.netcdf.zip')
    with zipfile.ZipFile(path_ERA5_raw+'download.netcdf.zip', 'r') as zip:
        zip.extractall(path_ERA5_raw)
    c.retrieve("reanalysis-era5-single-levels", {
            "product_type": ["reanalysis"],
            "variable": ["geopotential"],
            "year": ["2024"],
            "month": ["06"],
            "day": ["01"],
            "time": ["12:00"],
            "data_format": "netcdf"
        }, path_ERA5_raw+'era5_geopotential_pressure.nc')

In [ ]:
!ls $path_ERA5_raw/*.nc

In [ ]:
ds  = xr.open_dataset(path_ERA5_raw+'data_stream-moda.nc').drop_vars(("number", "expver"))
ds

In [ ]:
ds.rename({'valid_time': 'time'}).to_netcdf(path_ERA5_raw+"era5_monthly_averaged_data.nc")